# ⚖️ Class 10.7 — Cluster Evaluation Metrics & Ethics
### 301 – ML Techniques

**Learning Goals:** Calculate Silhouette, Davies-Bouldin, ARI, NMI. Complete RFM segmentation case study. Discuss ethics of postal-code segmentation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs, make_moons, make_circles
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 5)
print('✅ Ready')
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                              adjusted_rand_score, normalized_mutual_info_score)

---
## Part 1 — Evaluation Metrics Comparison

| Metric | Range | Better | Needs true labels? |
|---|---|---|---|
| Silhouette | −1 to 1 | Higher | No |
| Davies-Bouldin | ≥0 | Lower | No |
| ARI | −1 to 1 | Higher | Yes |
| NMI | 0 to 1 | Higher | Yes |

In [ ]:
# Generate data with known true clusters (for ARI/NMI)
from sklearn.datasets import make_blobs
X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)
X_s = StandardScaler().fit_transform(X)

algorithms = {
    'KMeans (k=4)':     KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_s),
    'KMeans (k=2)':     KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_s),
    'KMeans (k=6)':     KMeans(n_clusters=6, n_init=10, random_state=42).fit(X_s),
    'DBSCAN':           DBSCAN(eps=0.5, min_samples=5).fit(X_s),
    'Agglomerative(4)': AgglomerativeClustering(n_clusters=4).fit(X_s),
}

print(f'  {"Algorithm":<22} {"Silhouette":>12} {"Davies-Bouldin":>15} {"ARI":>8} {"NMI":>8}')
print('-'*68)
results = []
for name, model in algorithms.items():
    labels = model.labels_
    if len(set(labels[labels!=-1])) < 2:
        print(f'  {name:<22} (insufficient clusters)')
        continue
    valid = labels != -1
    sil = silhouette_score(X_s[valid], labels[valid]) if valid.sum() > 1 else 0
    dbi = davies_bouldin_score(X_s[valid], labels[valid])
    ari = adjusted_rand_score(y_true[valid], labels[valid])
    nmi = normalized_mutual_info_score(y_true[valid], labels[valid])
    print(f'  {name:<22} {sil:>12.3f} {dbi:>15.3f} {ari:>8.3f} {nmi:>8.3f}')
    results.append((name, sil, dbi, ari, nmi))

print()
print('QUESTIONS:')
print('1. Which algorithm performs best according to ARI (has true label ground truth)?')
print('   Answer: KMeans(k=4) and Agglomerative(4) — they match the true 4-cluster structure exactly,')
print('           so ARI ≈ 1.0.')
print('2. Do Silhouette and Davies-Bouldin agree with ARI? Why or why not?')
print('   Answer: Mostly yes for this dataset. Silhouette is highest and DBI lowest at k=4, which matches ARI.')
print('           They can disagree when clusters are not spherical, because Silhouette/DBI assume')
print('           compact, well-separated round clusters while ARI just compares to the true labels.')


---
## Part 2 — RFM Customer Segmentation Case Study

RFM = **Recency** (days since last purchase) + **Frequency** (number of purchases) + **Monetary** (total spend)

A common business use case for clustering.

In [ ]:
np.random.seed(42)
n_customers = 200
rfm = pd.DataFrame({
    'CustomerID' : range(1, n_customers+1),
    'Recency'    : np.concatenate([np.random.randint(1,30,50), np.random.randint(30,90,80),
                                    np.random.randint(90,365,70)]),
    'Frequency'  : np.concatenate([np.random.randint(10,50,50), np.random.randint(3,15,80),
                                    np.random.randint(1,5,70)]),
    'Monetary'   : np.concatenate([np.random.uniform(500,5000,50), np.random.uniform(100,800,80),
                                    np.random.uniform(10,200,70)]),
})

X_rfm = StandardScaler().fit_transform(rfm[['Recency','Frequency','Monetary']])
km_rfm = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_rfm)
rfm['Cluster'] = km_rfm.labels_

print('RFM cluster profiles:')
profile = rfm.groupby('Cluster')[['Recency','Frequency','Monetary']].mean().round(1)
print(profile)
print()

def label_rfm(r, f, m):
    if r < 30 and f >= 10 and m > 1000:
        return 'Champions', 'Reward with VIP perks; ask for referrals'
    if r < 60 and m > 500:
        return 'Loyal customers', 'Upsell related products; loyalty program'
    if r > 90 and f < 5:
        return 'At risk / Lost', 'Win-back campaign with discount'
    return 'Casual / new', 'Send onboarding offers and re-engagement emails'

print('Label each cluster:')
for c in range(4):
    row = profile.loc[c]
    name, action = label_rfm(row.Recency, row.Frequency, row.Monetary)
    print(f'  Cluster {c}: R={row.Recency:.0f}d, F={row.Frequency:.0f}x, M=${row.Monetary:.0f}  → Label: {name}')
    print(f'  Recommended action: {action}')


---
## Part 3 — Ethics Discussion

**Scenario:** A bank clusters customers by postal code and income to determine credit limits.
Clusters correlate with race due to historical segregation patterns.

In [ ]:
print('ETHICS CASE STUDY — READ CAREFULLY')
print()
print('A bank trains a K-Means model on customer data including postal code.')
print('The resulting clusters are used to set credit card limits.')
print('Analysis reveals: customers in predominantly minority postal codes')
print('are systematically placed in low-credit-limit clusters.')
print()
print('DISCUSSION QUESTIONS (write your answers):')
print()
print('1. Is this model "fair" if postal code was not intentionally used as a race proxy?')
print('   Answer: No. Intent does not matter — the *outcome* is discriminatory. Postal code is a')
print('           well-known proxy for race and socio-economic status because of historical')
print('           residential segregation. This is "disparate impact" and is illegal under most')
print('           anti-discrimination law regardless of intent.')
print()
print('2. What data should be excluded from clustering to prevent this outcome?')
print('   Answer: Direct protected attributes (race, religion, gender) and obvious geographic proxies')
print('           (postal code, neighbourhood). Names and surnames can also be proxies and should be excluded.')
print()
print('3. If you remove postal code but income correlates with it, does that solve the problem?')
print('   Answer: No. Removing one proxy is not enough — many features (income, occupation, education)')
print('           also correlate with race. You must audit the model output for disparate impact,')
print('           not just the inputs.')
print()
print('4. What audit steps should be run BEFORE deploying any customer segmentation model?')
print('   Answer: Compare cluster membership and outcomes (credit limit, approval rate) across protected')
print('           groups using fairness metrics (demographic parity, equal opportunity). Run an external')
print('           review and publish a model card describing data, intended use, and known limitations.')
print()
print('5. Under PIPEDA (Canadian privacy law) and GDPR, what rights do customers have')
print('   when automated decisions affect them?')
print('   Answer: The right to know an automated decision was used, the right to a meaningful')
print('           explanation of the logic involved, the right to a human review of the decision,')
print('           and (under GDPR Article 22) the right not to be subject to solely automated')
print('           decisions with significant effects.')


---
## ✅ Complete!

**Clustering metrics summary:**
- Silhouette and Davies-Bouldin: no true labels needed — intrinsic evaluation
- ARI and NMI: require true labels — extrinsic evaluation

**Ethics in clustering:**
Unsupervised models are NOT automatically fair.
Clusters can reproduce and amplify societal biases even without explicit protected attributes.